In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.SaltRemover import SaltRemover
from rdkit.Chem.FilterCatalog import *

<frozen importlib._bootstrap>:488: RuntimeWarning: to-Python converter for class boost::shared_ptr<class RDKit::FilterHierarchyMatcher> already registered; second conversion method ignored.


## Reading in the Data from Experiments

In [3]:
### Dose-Response Data
secondary_screen = pd.read_excel('PstP-SecondaryScreen-with-iterative-res.xlsx')

### Data of Retrieved Actives from CBWS_609
orig_actives = pd.read_csv('cbws609_actives_29.csv', header=0)

### Data from Docking Ranking Baseline
docking_screen = pd.read_csv('pstp_all_94044_clnsmi_murcko_iters_fps_clusters_umap_randiters_dockscores.csv', header=0)
##### Only selecting top 4000 as visualized in the paper ######
docking_screen = docking_screen[docking_screen['ECR_rank'] <= 4000]

### Raw Data from CDD export with controls
full_data = pd.read_csv('CDDexport-ALL.csv', header=0)

C:\Users\ryank\AppData\Local\Temp\ipykernel_9964\2034373233.py:8: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  docking_screen = pd.read_csv('pstp_all_94044_clnsmi_murcko_iters_fps_clusters_umap_randiters_dockscores.csv', header=0)


In [3]:
secondary_screen_actives = secondary_screen[secondary_screen['PstP Gitterlab Active'] == 1.0]
secondary_screen_inactives = secondary_screen[secondary_screen['PstP Gitterlab Active'] == 0]

In [4]:
secondary_screen_finds = secondary_screen.loc[((secondary_screen['PstP Gitterlab Active'] == 1.0) | (secondary_screen['PstP Gitterlab Active'] == 0))]

In [24]:
secondary_screen_finds.head()

,Molecule Name,Structure,CDD Number,SMILES,Supplier ID,PstP IC50 (uM),PstP Hill slope,PstP Maximum response (%),PstP R squared,PstP Curve class,PstP Phosphatase Dose Response: Dose-response Plot,IC50 >128,% Inhibition >20,Curve class 1.0-3.0,Follow-up,Strategy Iteration Recovered,PstP Gitterlab Active,PAINS
1,SMSSF-0552804,,NaN,F[B-](F)(F)F.CCCCN1C2=C(C3=CC=CC=C3C=C2)C(C)(C...,F9995-0477,7.7,2.28,38.3,0.925,1.2,,Y,Y,Y,Y,37.0,1.0,Y
5,SMSSF-0046890,,CDD-1745124,ClC1=C(Cl)C=C(C=C1)C1=CC=C(O1)\C=C\C(=O)\C=C/C...,F0344-1805,10.5,3.61,61.7,0.986,1.2,,Y,Y,Y,Y,7.0,1.0,Y
7,SMSSF-0046584,,CDD-1745060,BrC1=CC=C(C=C1)C1=CC=C(O1)\C=C\C(=O)\C=C/C1=CC...,F0344-1807,11.1,2.04,53.0,0.969,1.2,,Y,Y,Y,Y,6.0,1.0,Y
8,SMSSF-0061470,,CDD-1607558,OC1=C(C=CC2=C1N=CC=C2)C(NC1=NC=CS1)C1=CC=C(C=C...,F0842-0013,13.6,2.07,31.4,0.982,1.2,,Y,Y,Y,Y,19.0,1.0,N
9,SMSSF-0053303,,CDD-1746315,[O-][N+](=O)C1=CC=C(C=C1)C(=O)NN1C(=S)S\C(=C\C...,F3104-0060,14.6,2.11,47.3,0.987,1.2,,Y,Y,Y,Y,2.0,1.0,N


In [47]:
len(pd.unique(gitter_lab_secondary_fail_PAINS[gitter_lab_secondary_fail_PAINS['PstP Gitterlab Active'] == 1.0]['SMILES']))

9

## Moayad Notes

We merged these 4000 compounds with your 326 compound file. 
Here is a summary stats:
- Among the 326 compounds,  cbws_609 captured 43.
- Among the 126-hit compounds,  cbws_609 captured 26.
- Among the 40 followup compounds, cbws_609 captured 13.
- Among the 4 microbiologically active, cbws_609 captured 0.


## Docking Score

In [53]:
docking_screen_actives = docking_screen[docking_screen['PstP True Active'] == 1]
len(docking_screen_actives[docking_screen_actives['Passes PAINS Filter'] == 1])

9

In [14]:
temp_sorted = docking_screen.sort_values(by=['ECR_rank'])

start_value = 0
for index, row in temp_sorted.iterrows():
    if (row['ECR_rank'] != start_value + 1):
        print(row['ECR_rank'] - 1)
        start_value += 1
    start_value += 1

268.0
288.0
325.0
600.0
920.0
941.0
1122.0
1203.0
1428.0
1446.0
1509.0
1545.0
1581.0
1589.0
1917.0
2280.0
2438.0
2523.0
2536.0
2626.0
2796.0
2841.0
2842.0
2972.0
3198.0
3217.0
3777.0
3856.0


#### The above list are 28 ECR ranks that are missing from the top 4000 list of docking scores

In [35]:
secondary_screen_ids = list(secondary_screen['Molecule Name'])
docking_screen_ids = list(docking_screen['molid'])
secondary_screen_finds_ids = list(secondary_screen_finds['Molecule Name'])

overlap = []
for i in range(len(docking_screen_ids)):
    for j in range(len(secondary_screen_ids)):
        if(docking_screen_ids[i] == secondary_screen_ids[j]):
            overlap.append(docking_screen_ids[i])

In [25]:
len(overlap)

30

In [37]:
temp_overlap = []
for i in range(len(secondary_screen_finds_ids)):
    if (secondary_screen_finds_ids[i] in overlap):
        temp_overlap.append(secondary_screen_finds_ids[i])

len(temp_overlap)

8

In [32]:
follow_up_secondary_screen = secondary_screen.loc[((secondary_screen['IC50 >128'] == 'Y') & (secondary_screen['% Inhibition >20'] == 'Y' ) & (secondary_screen['Curve class 1.0-3.0'] == 'Y'))]

secondary_screen_ids = list(follow_up_secondary_screen['Molecule Name'])
docking_screen_ids = list(docking_screen['molid'])

overlap_2 = []
for i in range(len(docking_screen_ids)):
    for j in range(len(secondary_screen_ids)):
        if(docking_screen_ids[i] == secondary_screen_ids[j]):
            overlap.append(docking_screen_ids[i])

In [33]:
overlap_2

[]

# Testing PAINS Filter

In [14]:
### Defining PAINS filter 
params = FilterCatalogParams()
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_A)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_B)
params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS_C)
pains_catalog = FilterCatalog(params)

### Testing PAINS filter on Docking Baseline Data

In [51]:
## Just iterate over the secondary screen molecules and/or the docking screen 
docking_screen_PAINS = []
for smile in pd.unique(docking_screen['SMILES']):
    pains_filter = not pains_catalog.HasMatch(Chem.MolFromSmiles(smile))
    if(pains_filter == True):
        docking_screen_PAINS.append('Y')
    else:
        docking_screen_PAINS.append('N')


In [18]:
failures = []
i = 0
for index, row in docking_screen.iterrows():
    if(row['Passes PAINS Filter'] != docking_screen_PAINS[i]):
        failures.append(row['ECR_rank'])
    i += 1

In [20]:
len(failures)

17

## Testing how many molecules found in dose-response screening passed the PAINS filter

In [7]:
secondary_screen_PAINS = []
secondary_screen_PAINS_fails = []
for index, row in secondary_screen.iterrows():
    smile = row['SMILES']
    pains_filter = not pains_catalog.HasMatch(Chem.MolFromSmiles(smile))
    if(pains_filter == True):
        secondary_screen_PAINS.append('Y')
    else:
        secondary_screen_PAINS.append('N')
        secondary_screen_PAINS_fails.append(index + 2)

In [ ]:
### Showing distribution of molecules that passed and did not pass the PAINS filter
Counter(secondary_screen_PAINS)

In [ ]:
secondary_screen['PAINS'] = secondary_screen_PAINS
secondary_screen_finds = secondary_screen.loc[((secondary_screen['PstP Gitterlab Active'] == 1.0) | (secondary_screen['PstP Gitterlab Active'] == 0))]
Counter(secondary_screen_finds['PstP True'])

In [ ]:
### The metrics used by Nathan Wlodarchak to explore initial activity of molecules
secondary_screen_actives = secondary_screen.loc[((secondary_screen['% Inhibition >20'] == 'Y') & (secondary_screen['IC50 >128'] == 'Y')
                                                       & (secondary_screen['Curve class 1.0-3.0'] == 'Y'))]

### Separating the molecules that passed the PAINS and the ones that didn't
secondary_screen_actives_PAINS = secondary_screen_actives[secondary_screen_actives['PAINS'] == 'Y']
secondary_screen_actives_fail_PAINS = secondary_screen_actives[secondary_screen_actives['PAINS'] == 'N']

### Grouping the molecules that our CBWS_609 model found by whether or not they passed the PAINS filter
gitter_lab_secondary_PAINS = secondary_screen_actives_PAINS.loc[((secondary_screen_actives_PAINS['PstP Gitterlab Active'] == 1.0) |
                                                                 (secondary_screen_actives_PAINS['PstP Gitterlab Active'] == 0))]

gitter_lab_secondary_fail_PAINS = secondary_screen_actives_fail_PAINS.loc[((secondary_screen_actives_fail_PAINS['PstP Gitterlab Active'] == 1.0) |
                                                                 (secondary_screen_actives_fail_PAINS['PstP Gitterlab Active'] == 0))]

### Showing the molecules that passed the PAINS filter and that our CBWS_609 model predicted as active
gitter_lab_secondary_PAINS[gitter_lab_secondary_PAINS['PstP Gitterlab Active'] == 1.0]

## Testing how many actives found by CBWS passed the PAINS filter

In [7]:
orig_actives_PAINS = []
orig_active_PAINS_fails = []
for index, row in orig_actives.iterrows():
    smile = row['SMILES']
    pains_filter = not pains_catalog.HasMatch(Chem.MolFromSmiles(smile))
    if(pains_filter == True):
        orig_actives_PAINS.append('Y')
    else:
        orig_actives_PAINS.append('N')
        orig_active_PAINS_fails.append(index)

In [9]:
Counter(orig_actives_PAINS)

Counter({'Y': 20, 'N': 9})

## Showing Data Split Procedure

In [4]:
### Removed the control data from the CDD export (controls have an NaN for Molecule Name)
non_nan_full_data = full_data[full_data['Molecule Name'].notna() == True]
non_nan_full_data_smiles = non_nan_full_data['Structure (CXSMILES)']
non_nan_full_data.head()

,Molecule Name,Structure (CXSMILES),Batch Name,Run Date,Plate,Well,Control State,Raw OD 405nm,Raw OD 405nm Z'-factor,% inhibition (%)
2,SMSSF-0612944,CC1=CC=C(C=C1)C(=O)NC1=CC=C(CC2=CC=C(NC(=O)C3=...,1.0,2019-08-05,LC4-SMSF-Plate001,A03,NaN,1.18,0.94,2.84
3,SMSSF-0160349,ClC1=CC(=C(Cl)S1)C1=CSC(NC(=O)C2=CC=C(Br)S2)=N...,2.0,2019-08-05,LC4-SMSF-Plate001,A04,NaN,1.16,0.94,4.61
4,SMSSF-0612947,ClC1=CC=C(C=C1)C(=O)N1N=C(C[C@H]1C1=NC2=CC=CC=...,1.0,2019-08-05,LC4-SMSF-Plate001,A05,NaN,1.18,0.94,2.84
5,SMSSF-0117523,"NC1=NC(=CS1)C1=CC=C2OCCOC2=C1 |c:3,16,t:1,7,9|",2.0,2019-08-05,LC4-SMSF-Plate001,A06,NaN,1.18,0.94,2.84
6,SMSSF-0612953,COC(=O)NC(C#N)=C(NCC1=CC=CC=C1)NCC1=CC=CC=C1 |...,1.0,2019-08-05,LC4-SMSF-Plate001,A07,NaN,1.24,0.94,-2.46


### Looking at mean and standard devations of % inhibition

In [17]:
print('Mean of %% inhibition: %s '%(np.mean(non_nan_full_data['% inhibition (%)'].values)))
print()
print('Standard Deviation of %% inhibition: %s'%(np.std(non_nan_full_data['% inhibition (%)'].values)))
print()

Mean of % inhibition: -2.649699768454108 

Standard Deviation of % inhibition: 7.523700582195923



In [28]:
non_nan_smiles = []
for i in range(len(non_nan_full_data)):
    non_nan_smiles.append(list(non_nan_full_data['Structure (CXSMILES)'])[i].split('|')[0].replace(" ", ""))

In [32]:
non_nan_full_data['Parsed Smiles'] = non_nan_smiles
non_nan_full_data.head()

C:\Users\ryank\AppData\Local\Temp\ipykernel_14220\2622449658.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_nan_full_data['Parsed Smiles'] = non_nan_smiles


,Molecule Name,Structure (CXSMILES),Batch Name,Run Date,Plate,Well,Control State,Raw OD 405nm,Raw OD 405nm Z'-factor,% inhibition (%),Parsed Smiles
2,SMSSF-0612944,CC1=CC=C(C=C1)C(=O)NC1=CC=C(CC2=CC=C(NC(=O)C3=...,1.0,2019-08-05,LC4-SMSF-Plate001,A03,NaN,1.18,0.94,2.84,CC1=CC=C(C=C1)C(=O)NC1=CC=C(CC2=CC=C(NC(=O)C3=...
3,SMSSF-0160349,ClC1=CC(=C(Cl)S1)C1=CSC(NC(=O)C2=CC=C(Br)S2)=N...,2.0,2019-08-05,LC4-SMSF-Plate001,A04,NaN,1.16,0.94,4.61,ClC1=CC(=C(Cl)S1)C1=CSC(NC(=O)C2=CC=C(Br)S2)=N1
4,SMSSF-0612947,ClC1=CC=C(C=C1)C(=O)N1N=C(C[C@H]1C1=NC2=CC=CC=...,1.0,2019-08-05,LC4-SMSF-Plate001,A05,NaN,1.18,0.94,2.84,ClC1=CC=C(C=C1)C(=O)N1N=C(C[C@H]1C1=NC2=CC=CC=...
5,SMSSF-0117523,"NC1=NC(=CS1)C1=CC=C2OCCOC2=C1 |c:3,16,t:1,7,9|",2.0,2019-08-05,LC4-SMSF-Plate001,A06,NaN,1.18,0.94,2.84,NC1=NC(=CS1)C1=CC=C2OCCOC2=C1
6,SMSSF-0612953,COC(=O)NC(C#N)=C(NCC1=CC=CC=C1)NCC1=CC=CC=C1 |...,1.0,2019-08-05,LC4-SMSF-Plate001,A07,NaN,1.24,0.94,-2.46,COC(=O)NC(C#N)=C(NCC1=CC=CC=C1)NCC1=CC=CC=C1


### Removing SALTs from the molecules

In [33]:
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem.SaltRemover import SaltRemover

saltRemover = SaltRemover(defnFilename='Salts.txt')
rdkit_mols = non_nan_full_data['Parsed Smiles'].astype(str).apply((lambda x: Chem.MolFromSmiles(x)))
rdkit_mols = rdkit_mols.apply((lambda x: saltRemover.StripMol(x)))
non_nan_full_data['rdkit SMILES'] = rdkit_mols.apply((lambda x: Chem.MolToSmiles(x)))

C:\Users\ryank\AppData\Local\Temp\ipykernel_14220\1436566701.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  non_nan_full_data['rdkit SMILES'] = rdkit_mols.apply((lambda x: Chem.MolToSmiles(x)))


In [35]:
len(pd.unique(non_nan_full_data['rdkit SMILES']))

93948

In [37]:
non_nan_full_data['1024 MorganFP Radius 2'] = rdkit_mols.apply((lambda x: AllChem.GetMorganFingerprintAsBitVect(x, 
                                                                                                      radius=2, 
                                                                                                      nBits=1024).ToBitString()))

[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerator
[14:29:22] DEPRECATION WARNING: please use MorganGenerat

In [38]:
len(pd.unique(non_nan_full_data['1024 MorganFP Radius 2']))

93586